「AI Engineering Challenge ～煩雑な社内ドライブをハックせよ～」へようこそ!

このチュートリアルでは、LangChainを用いて基本的なRAGを構築します。

本チュートリアルでは、まずは「RAGを実行できること」を目標としてください。

なお、この段階で構築するRAGは、精度が十分でない可能性があります。場合によっては、すべて「わかりません」と回答する場合よりも悪い結果となることがあります。そのため、チュートリアル実施後は、RAGの精度の低さをどのように改善していくかが主な課題となります。

# ライブラリの読み込み
このチュートリアルでは、LangChainを用いて基本的なRAG（Retrieval-Augmented Generation）システムを構築します。

使用するLLMには、OpenAI APIのGPT-5.4を利用します。

なお、OpenAI APIの利用料金は従量課金制となっています。API利用料金については、参加者の皆様にてご負担いただきますようお願いいたします。

なお本チュートリアルが動作したライブラリの環境は以下となります。

ライブラリの不整合などのエラーが出た場合は、ご参照ください。

```
pandas==3.0.1
numpy==2.4.3
langchain==1.2.10
langchain-classic==1.0.2
langchain-community==0.4.1
langchain-core==1.2.23
langchain-openai==1.1.11
langchain-text-splitters==1.1.1
pypdf==6.8.0
python-pptx==1.0.2
unstructured==0.21.5
docx2txt==0.9
msoffcrypto-tool==6.0.0
chromadb==1.5.5
```

In [1]:
import os
import json
from pathlib import Path

In [ ]:
import pandas as pd
import numpy as np

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader, Docx2txtLoader, UnstructuredExcelLoader, UnstructuredPowerPointLoader

# apiキーの読み取り
tutorial.ipynbと同じディレクトリ(フォルダ)に`.env`ファイルを作成し、そこにopenaiのapiキーを以下のように入れて保存してください。

.envファイル

```.envファイル
OPENAI_API_KEY=APIキー
```

保存後以下のコード実行します。

※ APIキーの作成方法は、「OpenAI APIキー 作成方法」などのキーワードで生成AIサービスに質問するか、Web検索すると確認できます。

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
# apiキーを取得します
api_key = os.getenv("OPENAI_API_KEY")

# データ読み込み

まず本コンペで利用する質問データを確認してみます。

本コンペティションでは、以下の2つの質問データがありますが、

- `share/質問回答/questions_valid.csv`: 作成したRAGシステムの精度を検証する際にご利用ください。(こちらの質問内容は提出には含みません。)

- `share/質問回答/questions_test.csv`: 作成したシステムで回答していただき、最終的に投稿していただくための質問ファイルとなっています。(本チュートリアルをそのまま実行するとこちらの予測ファイルが作成されます。)

このチュートリアルのコードでは、評価データ(`questions_test.csv`)に対してRAGの予測を行います。コードをそのまま実行した場合は、`questions_test.csv` のデータに対して予測が行われる設定になっていますが、

以下コードの`validation`をTrueにすると、`questions_valid.csv`で検証を行うことができます。

データパスは以下を想定しています。


```
├──  tutorial.ipynb
├──  .env
├── share/
│   ├── 共有ドライブ/
│   │   ├── プロジェクト/
│   │   └── 社内管理/
│   └── 質問回答/
│       ├── questions_valid.csv
│       └── questions_test.csv
└── evaluation/
```

In [11]:
# ローカルの検証データで評価を行うか
validation = False

In [12]:
if validation:
    data = pd.read_csv("share/質問回答/questions_valid.csv")
else:
    data = pd.read_csv("share/質問回答/questions_test.csv")

In [14]:
data.head(3)

,index,question
0,0,白峰信用リスク評価の提案書old.pptxから提案書.pptxへの更新内容のうち、案件遂行に...
1,1,恒一会 かえで総合病院の最終報告書old版と最新版を比較したとき、案件遂行に関連する実質的な...
2,2,青嶺不動産アセットマネジメントのスケジュール_r2.xlsxにおいて、オレンジにハイライトさ...


In [15]:
data.shape

(100, 2)

# ドキュメントファイルの読み込み設定

RAGシステムを構築するには、まずドキュメントを検索しやすい状態にする必要があります。

コンピューターは人間のように文字を理解できないため、言葉や文字の情報を数値に置き換える必要があります。

RAGでは、この整理にあたる処理として、文章を「意味を表す数値（ベクトル）」へ変換して保存します。

質問が入力されると、その質問も同じようにベクトルへ変換し、意味が近いドキュメントを検索します。そして、見つかったドキュメントだけを生成AIへ渡し、その内容を参考に回答を生成します。

RAGのイメージ画像

![image](https://static.signate.jp/migration/1515-task_description-1.png)

LangChainでも、このような流れでRAGシステムを構築します。ただし、いきなりベクトル化を行うことはできません。

まずは、ベクトル化するための元となるドキュメントをプログラムで読み込む必要があります。

実際の業務では、PDFやWord、PowerPointなど、さまざまな形式のファイルを扱い、通常はアイコンをクリックすることで、ドキュメントを開くことはできます。ただしプログラミングではそのような方法は利用できません。それぞれのファイル形式に対応した読み込み処理が必要になります。

LangChainには、このようなドキュメントを読み込むための**Document Loader（ローダー）**が用意されています。本チュートリアルでは、このローダーを利用してドキュメントを読み込んでいきます。

読み込むべきドキュメントは、「共有ドライブ」フォルダ内に格納されているのでそちらを読み込みます。

In [16]:
# ドキュメントの一覧(共有ドライブ内のファイルをすべて)を取得
dir_path = Path("share/共有ドライブ/")

# ディレクトリ以下のすべてのファイルパスを取得
file_paths = [p for p in dir_path.rglob("*") if p.is_file() and p.name != ".DS_Store"]

print(len(file_paths))
print(file_paths[:5])

416
[WindowsPath('share/共有ドライブ/社内管理/データアステル社内管理_決裁基準.md'), WindowsPath('share/共有ドライブ/社内管理/データアステル社内規定_パスワード導出規則.docx'), WindowsPath('share/共有ドライブ/社内管理/座席表.pptx'), WindowsPath('share/共有ドライブ/社内管理/社内用語集.docx'), WindowsPath('share/共有ドライブ/プロジェクト/京橋信用ソリューションズ株式会社/00.提案/提案書_final.pptx')]


In [17]:
# train.csv / train.xlsx はデータ量が多いため、先頭20行のみ読み込む特別処理
TARGET_HEAD_FILES = {"train.csv", "train.xlsx"}

# train.csv, train.xlsxの判定を行う関数
def should_load_head_only(file_path):
    return file_path.name.lower() in TARGET_HEAD_FILES

In [18]:
# ファイル読み込み(langchainのローダーを利用)
def load_docs(file_path):
    ext = file_path.suffix.lower()

    try:
        if ext in [".txt", ".md", ".py", ".json", ".jsonl", ".yml", ".yaml"]:
            loaded = TextLoader(str(file_path), encoding="utf-8").load()
        elif ext == ".docx":
            loaded = Docx2txtLoader(str(file_path)).load()
        elif ext == ".csv":
            loaded = CSVLoader(str(file_path), encoding="utf-8").load()
            # train.csvの場合
            if should_load_head_only(file_path):
                loaded = loaded[:20]
            else:
                loaded = CSVLoader(str(file_path), encoding="utf-8").load()
        elif ext == ".pdf":
            loaded = PyPDFLoader(str(file_path)).load()
        elif ext in {".xlsx", ".xls"}:
            loaded = UnstructuredExcelLoader(str(file_path)).load()
            # train.xlsxの場合
            if should_load_head_only(file_path):
                loaded = loaded[:20]
                
        elif ext in {".pptx", ".ppt"}:
            loaded = UnstructuredPowerPointLoader(str(file_path)).load()
        # どのファイルにも該当しないときは一旦無視する
        else:
            print(f"どのファイルにも該当しませんでした。: {file_path}")
            loaded = None
    except Exception as e:
        print(f"skip: {file_path} ({e})")
        loaded = None

    return loaded

`共有ドライブ`内のファイルを全て読み込みます

In [19]:
all_files = []
for file_path in file_paths:
    if not file_path.exists():
        print(f'[skip] not found: {file_path}')
        continue

    # ドキュメント読み込み
    doc = load_docs(file_path)
    
    if doc:
        # メタデータを付与(ファイル名とデータパス)
        for d in doc:
            # 絶対パスを取得する
            #d.metadata["source_path"] = str(file_path.resolve())
            d.metadata["file_name"] = str(file_path.name)
    else:
        continue
        
    all_files.extend(doc)
print("FINISH!")

skip: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\02.計画\スケジュール.xlsx (No module named 'networkx')
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\pyproject.toml
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\uv.lock
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\notebooks\01_eda.ipynb
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\reports\figures\categorical_distribution_top3.png
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\reports\figures\feature_correlation_heatmap.png
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\reports\figures\figure_06.png
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\reports\figures\missing_rate_top20.png
どのファイルにも該当しませんでした。: share\共有ドライブ\プロジェクト\京橋信用ソリューションズ株式会社\04.分析\analysis_project\reports\figures\numeric_distribution_top6.png
ど

PNGファイル、Jupyter Notebook（`.ipynb`）ファイル、Pythonのキャッシュファイル（`.pyc`）などは、ドキュメントとして読み込めませんでした。

また、パスワードが設定されているファイルについても、本チュートリアルでは読み込み対象外としています。

このチュートリアルではこれらのファイルは扱いませんが、必要に応じて、各ファイル形式に対応した読み込み処理を追加してもよいでしょう。

In [20]:
# 読み込まれたファイルの数を確認
len(all_files)

1119

読み込まれたデータ数がファイル数よりも多くなっています。

これは一部のLangChainのローダーの仕様によるものです。

In [21]:
# 分割されて読まれているファイルの例
print(all_files[11].metadata)
print(all_files[12].metadata)

{'source': 'share\\共有ドライブ\\プロジェクト\\京橋信用ソリューションズ株式会社\\03.データ\\train.csv', 'row': 4, 'file_name': 'train.csv'}
{'source': 'share\\共有ドライブ\\プロジェクト\\京橋信用ソリューションズ株式会社\\03.データ\\train.csv', 'row': 5, 'file_name': 'train.csv'}


In [22]:
# 一部中身も確認してみましょう(表示が長いため一部省略しています)
print("メタデータ")
print(all_files[0].metadata)
print("中身")
print(all_files[0].page_content[:100])

メタデータ
{'source': 'share\\共有ドライブ\\社内管理\\データアステル社内管理_決裁基準.md', 'file_name': 'データアステル社内管理_決裁基準.md'}
中身
# データアステル社内管理_決裁基準

## 1. 目的

本規程は、案件の契約金額および契約条件に応じた社内決裁レベルを定め、提案・契約・請求に関する承認プロセスを統一することを目的とする。

##


# テキストのベクトル化

次に、読み込んだテキストをベクトル化します。

テキストのベクトル化には、TF-IDFやEmbeddingなどの方法がありますが、本チュートリアルではOpenAI APIのEmbeddingモデルを利用します。

※ OpenAI APIの利用料金は、実行者の負担となりますのでご注意ください。

In [23]:
# ベクトルデータの保存先
save_dir = f"sample_vector"
# ベクトルデータの保存先を識別するための名前（本チュートリアルでは "sample" を使用します。）
collection_name = "sample"

In [24]:

# ファイルがすでに存在する場合は、ベクトルファイルを作り直さない。
if os.path.exists(save_dir):
    vs = Chroma(
    persist_directory=save_dir,
    embedding_function=OpenAIEmbeddings(api_key=api_key, model="text-embedding-3-large"),
    collection_name=collection_name
    )
    count = vs._collection.count()
    
    if count == 0:
        raise RuntimeError(
            "Chroma DBのディレクトリは存在しますが、中身が空です。"
            "save_dirを削除して作り直してください。"
        )
    # チャンクの確認
    print(f"chunk size: {count}")
else:
    # 2048文字以上のドキュメントがある場合、2048トークンごとにテキストを分割します。分割後も前後の文脈を取得するために256トークン分の重複を許します。
    splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=256)
    chunks = splitter.split_documents(all_files)
    
    if len(chunks) == 0:
        raise ValueError("chunkサイズが0件です。all_filesの読み込み結果を確認してください。")

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=OpenAIEmbeddings(api_key=api_key, model="text-embedding-3-large"),
        collection_name=collection_name,
        persist_directory=save_dir)
    print(f"chunk size: {len(chunks)}")

chunk size: 1407


以下のエラーが出た場合は、作成されたディレクトリを削除して再実行してください。

```
Chroma DBのディレクトリは存在しますが、中身が空です。

save_dirを削除して作り直してください。
```

以下のエラーが出た場合は、`dir_path`の指定がうまくいっているか確認してください。

```
chunkサイズが0件です。all_filesの読み込み結果を確認してください。
```


作成したら、中身どうなっているのか確認します。

In [25]:
res = vs._collection.get(limit=5, include=["embeddings", "documents", "metadatas"])

print(res['ids'])          # IDのリスト
print(res['metadatas'])    # メタデータ（ファイル名やページ番号など）
print(res['documents'])    # ドキュメントの中身
print(res['embeddings'])   # ベクトルの中身

['ca10ebaa-11f5-4984-af9d-9247d2dcb30f', 'e0b546de-c2f1-4cc3-8a90-b2a829766648', '31d63bf4-79e8-48bf-a140-d174af51b4cf', '751dd8b1-5cd6-4f55-a99c-e6be887664bd', '73687591-79b1-4f8d-ae2b-285a4d95c1b4']
[{'file_name': 'データアステル社内管理_決裁基準.md', 'source': 'share\\共有ドライブ\\社内管理\\データアステル社内管理_決裁基準.md'}, {'file_name': 'データアステル社内規定_パスワード導出規則.docx', 'source': 'share\\共有ドライブ\\社内管理\\データアステル社内規定_パスワード導出規則.docx'}, {'source': 'share\\共有ドライブ\\社内管理\\社内用語集.docx', 'file_name': '社内用語集.docx'}, {'file_name': '社内用語集.docx', 'source': 'share\\共有ドライブ\\社内管理\\社内用語集.docx'}, {'source': 'share\\共有ドライブ\\社内管理\\社内用語集.docx', 'file_name': '社内用語集.docx'}]
['# データアステル社内管理_決裁基準\n\n## 1. 目的\n\n本規程は、案件の契約金額および契約条件に応じた社内決裁レベルを定め、提案・契約・請求に関する承認プロセスを統一することを目的とする。\n\n## 2. 通常の決裁基準\n\n契約金額（税込）に応じた基本の決裁レベルは次の通りとする。\n\n| 契約金額（税込） | 必要な承認 |\n|---|---|\n| 3,000,000円未満 | 主任承認 |\n| 3,000,000円以上 5,000,000円未満 | 課長承認 |\n| 5,000,000円以上 8,000,000円未満 | 部長承認 |\n| 8,000,000円以上 | 本部長承認 |\n\n## 3. 追加基準\n\n### 3.1 医療案件\n\n医療機関、医療法人、病院、診療所その他これに

確認できたら必要に応じて検索テストを行います

In [26]:
results = vs.similarity_search_with_score(
    query="契約",
    k=5
)

for r, score in results:
    print("score:", score)
    print(r.metadata.get("file_name"))
    print(r.page_content[:20])
    print("==========")

score: 1.217613697052002
契約書.docx
5. 契約期間

本契約の締結日および効
score: 1.2300786972045898
契約書.docx
本業務により新たに作成された成果物のうち
score: 1.2361257076263428
契約書.docx
乙は、再委託先に対して本契約と同等以上の
score: 1.2406445741653442
契約書.docx
8. 秘密保持

甲および乙は、本契約に
score: 1.245604157447815
社内用語集.docx
データアステル社内用語集

本資料は、社


契約に関するドキュメントが抜き出せていることが確認できました。

# RAGによる回答文生成

質問文に関連するドキュメントを取得できたので、次に取得した情報をもとに質問への回答を生成します。

In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

### プロンプトの設定

回答と根拠を出力してもらいます。

根拠は提出に不要ですが、間違えた回答のについては、回答理由を参照することによって、より正確な回答に修正するための手掛かりとなります。

In [28]:
prompt = ChatPromptTemplate.from_template("""
                                          あなたはドキュメントQAのアシスタントです。
                                          以下のコンテキストだけを根拠に、質問へ日本語で回答してください。
                                          
                                          # 回答条件
                                          - 回答は#出力JSONに従うものとします。
                                          - 与えられた情報を元に確実な回答が難しい場合は answer を「わかりません」にしてください。
                                          - 与えられていない情報から推論を行なって回答することは絶対にしないでください。
                                          - また回答(answer)は質問文対する回答にのみとします。回答に至った経緯はresaonに記入してください。
                                          
                                          # 質問
                                          {question}
                                          
                                          # コンテキスト
                                          {context}
                                          
                                          # 出力JSON:
                                          {{"answer": "回答", "reason": "根拠"}}
                                          """)

In [29]:
answer_list = [] # 回答結果を入れておくリスト
# context_list = [] # 候補となったドキュメントを入れておくリスト(このチュートリアルでは引用情報を保存しない。)
usage_list = [] # api利用状況を入れておくリスト
for i, d in data.iterrows():
    question = d["question"]
    
    # 候補の作成
    results = vs.similarity_search(
        query=question,
        k=5
    )

    # 検索結果を1つのコンテキストにまとめる
    context = "\n\n---\n\n".join(
        [f"[source: {r.metadata.get('source')}]\n{r.page_content}" for r in results]
        )
    
    # gpt5.4で回答生成
    llm = ChatOpenAI(api_key=api_key,model="gpt-5.4", temperature=0)
    chain = prompt | llm

    llm_answer = chain.invoke({"question": question, "context": context})
    print(f"質問: {question}")
    print(f"回答(LLM): {llm_answer.content}")

    print("=====")
    answer_json = json.loads(llm_answer.content)
    answer_list.append(answer_json)
    # context_list.append(context)
    usage_list.append(llm_answer.response_metadata['token_usage'])
print("FINISH!")

質問: 白峰信用リスク評価の提案書old.pptxから提案書.pptxへの更新内容のうち、案件遂行に関連する実質的な変更を挙げてください。
回答(LLM): {"answer":"提案書old.pptxから提案書.pptxへの更新内容のうち、案件遂行に関連する実質的な変更として確認できるのは「4.2 前処理方針策定」の追加です。具体的には、①欠損補完ルール策定、②外れ値処理の比較検討、③モデル別の前処理方針分離、④高欠損列の投入比較、⑤再現性確保のための条件固定、が追加されています。その他の掲載箇所（4.3以降、5〜11など）は、提示された範囲ではoldと新で同一内容です。","reason":"新しい提案書には「4.2 前処理方針策定」があり、列ごとの業務意味を踏まえた欠損補完、Winsorize/クリッピング等の外れ値処理比較、線形モデル向けと木系モデル向けの前処理分離、高欠損列の投入有無比較、前処理条件・評価条件の固定化による再現性確保が明記されています。一方、oldでは4.3から始まっており、同スライド相当の記載が確認できません。加えて、4.3〜11についてはコンテキスト上、新旧で文言が同一です。"}
=====
質問: 恒一会 かえで総合病院の最終報告書old版と最新版を比較したとき、案件遂行に関連する実質的な変更を挙げてください。
回答(LLM): {"answer":"わかりません","reason":"コンテキスト上、old版と最新版の内容を比較すると、契約期間、実績工数、請求金額、目的、スコープ、実施方法、特徴量選定、モデル比較結果などの記載内容は実質的に同一であり、確認できる差分は主にレイアウトや表記形式（改行や箇条書き表示）に見えます。案件遂行に関連する実質的な変更点があったと断定できる情報は与えられていません。"}
=====
質問: 青嶺不動産アセットマネジメントのスケジュール_r2.xlsxにおいて、オレンジにハイライトされている行のタスク名をすべて答えてください。
回答(LLM): {"answer":"わかりません","reason":"質問は『青嶺不動産アセットマネジメントのスケジュール_r2.xlsxにおいて、オレンジにハイライトされている行のタスク名』を求めていますが、提供されたコンテキストには当該Excelファイルの内容や、オ

「わかりません」が多くを占めますが、一部問題には回答しているようです。

# 投稿ファイルの作成

In [31]:
output_query = pd.DataFrame(answer_list)

In [32]:
output_query.head()

,answer,reason
0,提案書old.pptxから提案書.pptxへの更新内容のうち、案件遂行に関連する実質的な変更...,新しい提案書には「4.2 前処理方針策定」があり、列ごとの業務意味を踏まえた欠損補完、Win...
1,わかりません,コンテキスト上、old版と最新版の内容を比較すると、契約期間、実績工数、請求金額、目的、スコ...
2,わかりません,質問は『青嶺不動産アセットマネジメントのスケジュール_r2.xlsxにおいて、オレンジにハイ...
3,わかりません,質問は「恒一会 かえで総合病院の契約書」における太字箇所の抽出を求めていますが、提供されたコ...
4,わかりません,提示されたコンテキストには、蒼泉会 ひがし丘総合病院の01_eda.ipynb自体の内容や、...


In [34]:
# どのドキュメントの内容が引用されたかを確認したい場合は、必要に応じてコメントアウトを外してください
#output_query["context"] = context_list

In [35]:
# 回答に改行があるとエラーが出るので、改行がある場合は削除する

output_query["answer"] = output_query["answer"].apply(lambda x: x.replace("\n", " ").replace("\r", " "))

In [38]:
# 投稿ファイルにはindexと回答文のみの記載があれば良い
if validation:
    output_query["answer"].to_csv("predictions_valid.csv", header=None, encoding="utf-8")
else:
    # 投稿ファイル名は必ずpredictions.csvにしてください。
    output_query["answer"].to_csv("predictions.csv", header=None, encoding="utf-8")

In [29]:
# 回答の詳細も確認したい場合は、理由も含めて保存すると確認がしやすい
#output_query.to_csv("predictions.csv", header=None)

これで投稿ファイルの作成は完了です。

作成したpredictions.csvをzipファイルにすることで投稿できます。

※1 csvのファイル名は必ず`predictions.csv`にしてください

※2 zipファイルの名前は任意ですが、英数字、アンダースコア(_)、ハイフン(-)のみを使い、ドット(.)のあとに拡張子を英数字のみで入力してください

以下のコードでzipファイル化できますが、PC上のzipファイル作成でも問題ございません。

In [39]:
# 投稿ファイルをzipファイル化する
import zipfile
if not validation:
    with zipfile.ZipFile("tutorial_submit.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("predictions.csv")

本チュートリアルをそのまま実行して投稿した場合、スコアはおおよそ-0.1程度になる想定です。

In [40]:
pred = pd.read_csv("predictions.csv", header=None)

In [41]:
pred

,0,1
0,0,提案書old.pptxから提案書.pptxへの更新内容のうち、案件遂行に関連する実質的な変更...
1,1,わかりません
2,2,わかりません
3,3,わかりません
4,4,わかりません
...,...,...
95,95,未着手から完了への変更を除くと、案件遂行に関連する変更点として確認できるのは以下です。1) ...
96,96,T14/T15
97,97,わかりません
98,98,わかりません


# Next Action

本チュートリアルでは、LangChainを用いて基本的なRAGシステムを構築しました。

ただし、現時点の実装には、まだ読み込めていないファイルが存在したり、質問に対して適切な情報を取得できない場合があります。

以下に、本チュートリアルで構築したRAGシステムの主な課題を示します。必要に応じて、どのように改善できるか検討してみてください。

- PNGファイル、Jupyter Notebook（.ipynb）ファイル、パスワード付きファイルを読み込めていない
- WordファイルやPowerPointファイルに含まれる太字、文字色などの書式情報を取得できていない
- PDFファイルに含まれる画像や図表などの情報を取得できていない
- 検索対象となるファイルが非常に多い
  - ヒント: 質問文の中には、具体的なファイル名が指定されているものがあります
  - ヒント: ファイル名が指定されていなくても、企業名などの情報から検索対象を絞り込める可能性があります
- 一部の質問では、テキストを読み込むだけでなく、追加の処理や推論が必要になる可能性があります

またembeddingを使った方法以外にも、必要に応じてエージェントを活用することによって、より高い精度を目指せる可能性があります。

### 注意事項

本コンペティションにおけるRAGの実装は、新しい案件フォルダ、ファイル、質問を入れても同様の精度で回答できるロジックとすることが条件です。

このコードを改善して実装を進める場合でも、各質問に対して、**機械的な処理で入力から回答が得られる前提とした**実装をし、**新しい案件フォルダ、ファイル、質問が追加された場合でも、同様の精度で回答できる汎用的なロジックとなるような実装していただく**ようお願いいたします。

(詳しくはコンペティションページの「ルール」をご参照ください。)

# おまけ

## 正誤チェック

`question_valid` ファイルを利用している場合は、ローカル環境でスコアを評価できます。

評価コードは `evaluation.zip` に含まれています。ローカルで評価を行う場合は、`evaluation.zip` を解凍し、同梱されている `readme.md` の手順に従って環境を準備したうえで、評価コマンドを実行してください。

本チュートリアルでは、Notebook上から評価コマンドを実行し、評価結果を確認する方法を紹介します。

`readme.md` の手順は、実際の評価環境を再現するために用意されています。評価環境にできるだけ近い状態で実行したい場合は、Dockerを含めた仮想環境を利用することをおすすめします。

一方で、動作確認を行うだけであれば、実行に必要なライブラリを各自の環境にインストールして実行しても問題ありません。

ただし、環境やライブラリのバージョン、LLMの出力のばらつきなどにより、挙動や評価結果が完全には一致しない可能性があります。

In [ ]:
#import shutil
# 生成した回答結果をsubmitフォルダに移動します
#shutil.copy("predictions_valid.csv", "evaluation/submit/predictions.csv")

'evaluation/submit/predictions.csv'

In [ ]:
# 必要なライブラリが揃っていることを確認したら、以下のコマンドで評価を実行してください。
#!python evaluation/crag.py --result-name predictions.csv --result-dir evaluation/submit --ans-dir evaluation/data


Format Setting:
  time elapsed: 0.001596212387084961[s]

Validation:
  Checking data...Done
  Checking samples...Done
  Checking dtype...Done
  Checking tokens...Done
  time elapsed: 0.09291410446166992[s]

Loading Ground Truth:
  db shape: (30, 1)
  time elapsed: 0.00043487548828125[s]

Evaluation:
  By CRAG
  llm: gpt-5.2-2025-12-11
30it [00:28,  1.04it/s]
  time elapsed: 28.785815954208374[s]


Saving the Results:
  Score: -0.13333333333333333
  time elapsed: 0.011245965957641602[s]


## 回答までにかかった費用の計算

In [31]:
usage_df = pd.DataFrame(usage_list)

In [32]:
input_fee = (usage_df["prompt_tokens"].sum()/1000000)*2.50
output_fee = (usage_df["completion_tokens"].sum()/1000000)*15

In [33]:
(input_fee+output_fee)*160

np.float64(228.95480000000003)

1ドル=160円で換算した場合、本チュートリアルのコードをGPT-5.4で実行すると、約250円のAPI利用料金が発生します。（Embeddingの利用料金は含まれていないため、実際の費用はこれよりも高くなります。）

コストを意識して開発を進める場合は、APIの利用回数や消費トークン数を記録しておくことをおすすめします。これにより、実行時の費用を見積もったり、コスト削減の効果を確認したりしやすくなります。